# Option 3: Stock Clustering Analysis

## Objective
Group stocks into clusters based on financial characteristics

## Clustering Methods
- K-Means Clustering (optimized with Elbow Method)
- Hierarchical Clustering
- Analysis of cluster characteristics and risk profiles

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn scipy kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.distances import cdist
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
from sklearn.metrics import silhouette_score, davies_bouldin_score
import kagglehub, os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All libraries loaded successfully")

## Step 1: Data Loading & Selection

In [ ]:
# Download S&P 500 dataset
path = kagglehub.dataset_download("camnugent/sandp500")
files = os.listdir(path)
csv_file = [f for f in files if f.endswith(".csv")][0]
data = pd.read_csv(os.path.join(path, csv_file))

print(f"Dataset Shape: {data.shape}")
print(f"Unique stocks: {data['Name'].nunique()}")
print(f"Date Range: {data['date'].min()} to {data['date'].max()}")

# Get latest data for each stock
latest_data = data.sort_values('date').drop_duplicates('Name', keep='last')
print(f"\nStocks in dataset: {len(latest_data)}")

## Step 2: Feature Engineering for Clustering

In [ ]:
def create_clustering_features(stock_data):
    """Create features for stock clustering"""
    stock_data = stock_data.sort_values('date').reset_index(drop=True)
    
    # Volatility - standard deviation of returns
    stock_data['volatility'] = stock_data['close'].pct_change().rolling(20).std()
    
    # Average returns
    stock_data['avg_return'] = stock_data['close'].pct_change().rolling(20).mean()
    
    # Trading volume characteristics
    stock_data['avg_volume'] = stock_data['volume'].rolling(20).mean()
    stock_data['volume_volatility'] = stock_data['volume'].rolling(20).std() / stock_data['volume'].rolling(20).mean()
    
    # Price momentum
    stock_data['momentum'] = (stock_data['close'] - stock_data['close'].shift(20)) / stock_data['close'].shift(20)
    
    # Price range (high-low spread)
    stock_data['price_range'] = (stock_data['high'] - stock_data['low']) / stock_data['close']
    
    # Trend strength (higher close vs low)
    stock_data['trend_strength'] = (stock_data['close'] - stock_data['low']) / (stock_data['high'] - stock_data['low'])
    
    # Price level (normalized by 100s)
    stock_data['price_level'] = stock_data['close'] / 100
    
    # Beta-like volatility vs overall market
    stock_data['volume_to_price_ratio'] = stock_data['volume'] / stock_data['close']
    
    return stock_data

# Apply feature engineering to all stocks
data_processed = pd.concat([create_clustering_features(data[data['Name'] == stock]) 
                            for stock in data['Name'].unique()], ignore_index=True)

# Get latest record for each stock
clustering_data = data_processed.sort_values('date').drop_duplicates('Name', keep='last')
clustering_data = clustering_data.dropna()

print(f"Stocks with complete features: {len(clustering_data)}")
print(f"\nFeature Statistics:")
print(clustering_data[['volatility', 'avg_return', 'momentum', 'price_level']].describe().round(4))